# Google Colab Setup

Run vehicle tracking and make/model census on Google Colab T4 GPU.

## 1. Get Car-Census code and data

In [ ]:
from google.colab import userdata
import os

# 1. Retrieve the secret securely
token = userdata.get('GITHUB_TOKEN')

# 2. Define the repo details
# Note: Remove 'https://' from the start of your repo string for the formatting below
repo_path = "github.com/DmitryMatv/Car-Census.git"
repo_url = f"https://{token}@{repo_path}"

# 3. Clone the repo
!git clone {repo_url}
    
# 4. Change directory to the cloned repo
os.chdir('Car-Census')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/Car-Census/input_data
#!cp -rn /content/drive/MyDrive/input_data/. /content/Car-Census/input_data/

In [ ]:
!cp -rn /content/drive/MyDrive/input_data/IMG_5383_1440.mp4 /content/Car-Census/input_data/

In [ ]:
!cp -rn /content/drive/MyDrive/input_data/IMG_5386_1440.mp4 /content/Car-Census/input_data/

## 2. Install Dependencies

In [ ]:
%pip install -e .

## 3. Verify GPU, RF-DETR & FFmpeg

In [ ]:
import torch
import rfdetr

print(f"PyTorch version: {torch.__version__}")
print(f"RF-DETR package: {rfdetr.__file__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
!ffmpeg -hide_banner -encoders | grep nvenc || true

!ffmpeg -hide_banner -loglevel error \
  -f lavfi -i color=size=640x360:rate=1:duration=1 \
  -frames:v 1 -an -c:v h264_nvenc -f null -

In [ ]:
!ffmpeg -hide_banner -encoders | grep nvenc

## 3b. Batch Tests

In [ ]:
from collections.abc import Mapping, Sequence

from rfdetr import RFDETRSmall


def class_name_sample(class_names, limit=8):
    if isinstance(class_names, Mapping):
        return dict(list(class_names.items())[:limit])
    if isinstance(class_names, Sequence) and not isinstance(class_names, str):
        return dict(enumerate(class_names[:limit]))
    return {}

model = RFDETRSmall()
print("model:", type(model).__name__)
print("class_names sample:", class_name_sample(model.class_names))

In [ ]:
import time, subprocess, shlex

cmd = """
Car-Census analyze input_data/YtRoadTraffic_1440_30s.mp4 \
  --accelerator colab-t4 \
  --verbose
"""

start = time.perf_counter()
result = subprocess.run(shlex.split(cmd), text=True, capture_output=True)
elapsed = time.perf_counter() - start

print(result.stdout)
print(result.stderr)
print(f"elapsed_seconds={elapsed:.2f}")
print(f"video_seconds=30")
print(f"analysis_speed={30 / elapsed:.2f}x realtime")

In [ ]:
from config import build_effective_config
from pathlib import Path

config = build_effective_config(Path("."))
print("analysis.batch_size:", config.analysis.batch_size)
print("detector.model:", config.detector.model)
print("detector.input_size:", config.detector.input_size)

In [ ]:
from pathlib import Path
from config import build_effective_config
from detectors.factory import create_detector
from utils.video import iter_sampled_frames

config = build_effective_config(Path("."))

detector = create_detector(config, Path("."))

frames = []
for _, _, frame in iter_sampled_frames(
    Path("input_data/YtRoadTraffic_1440_30s.mp4"),
    source_fps=config.video.fps,
    target_fps=config.analysis.fps,
):
    frames.append(frame)
    if len(frames) == config.analysis.batch_size:
        break

print("requested batch:", len(frames))
print("detector diagnostics before:", detector.detection_diagnostics())

detections = detector.detect_batch(frames)
print("returned detection sets:", len(detections))
print("detections per frame:", [len(x) for x in detections])
print("detector diagnostics after:", detector.detection_diagnostics())

## 4. Set Environment Variables

In [ ]:
traffic_eye_api_key = userdata.get('TRAFFICEYE_API_KEY')
if not traffic_eye_api_key:
    raise RuntimeError('Add TRAFFICEYE_API_KEY to Colab Secrets before running classification.')

os.environ['TRAFFICEYE_API_KEY'] = traffic_eye_api_key
print('TRAFFICEYE_API_KEY loaded from Colab Secrets.')

## 5. Run Car-Census

### Option A: Run Full Pipeline (detect, track, classify, render)

In [ ]:
!Car-Census run input_data/YtRoadTraffic_1440_30s.mp4 --accelerator colab-t4

### Option B: Run Without Video Render (it's slow on Google Colab)

In [ ]:
!Car-Census run input_data/1_1440.MP4 --accelerator colab-t4 --skip-render

### Option C: Run with Camera ID

In [ ]:
!Car-Census run input_data/test4K.MP4 --camera-id my-camera --accelerator colab-t4

### Option D: ROI Edit (define polygon zone)

In [ ]:
!Car-Census roi edit input_data/test4K.MP4 --camera-id my-camera --device cuda

## 6. Get Results

In [ ]:
# Upload output folder directly to Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/output
!cp -rn /content/Car-Census/output/. /content/drive/MyDrive/output/

In [ ]:
# Download output folder directly
from google.colab import files

!zip -r output.zip output/
files.download('output.zip')

In [ ]:
# Download specific file
files.download('/content/car-census/outputs/<run-id>/annotated.mp4')